# Persian MMS/VITS fine-tuning on Kaggle

This notebook fine-tunes Meta's `facebook/mms-tts-fas` checkpoint on Persian speech. Before running it, enable **GPU** and **Internet** in Kaggle's Notebook options. The default configuration uses 1,000 ManaTTS samples for a practical first run.

> License note: Meta MMS weights are CC BY-NC 4.0, so the resulting weights are for non-commercial use.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/melolikepz/persian-mms-tts-kaggle.git'
REPO_DIR = Path('/kaggle/working/persian-mms-tts-kaggle')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Repository:', REPO_DIR)
print('Python:', sys.version)

In [ ]:
# Install versions known to work with the upstream training recipe.
%pip install -q -r requirements-kaggle.txt

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. In Kaggle, open Notebook options and select a GPU accelerator.')
print('GPU:', torch.cuda.get_device_name(0))
(REPO_DIR / 'monotonic_align' / 'monotonic_align').mkdir(exist_ok=True)
subprocess.run(
    [sys.executable, 'setup.py', 'build_ext', '--inplace'],
    cwd=REPO_DIR / 'monotonic_align',
    check=True,
)

## Training settings

Keep the defaults for the first end-to-end run. For a longer run, increase `TRAIN_SAMPLES` and `EPOCHS`. The first 25 records are reserved for evaluation.

In [ ]:
import json

TRAIN_SAMPLES = 1000
EPOCHS = 20
BATCH_SIZE = 8
OUTPUT_DIR = Path('/kaggle/working/persian-mms-tts')
BASE_TRAIN_DIR = Path('/kaggle/working/mms-tts-fas-train')

template_path = REPO_DIR / 'training_config_examples/finetune_mms_fa_kaggle.json'
config = json.loads(template_path.read_text(encoding='utf-8'))
config['train_split_name'] = f'train[25:{25 + TRAIN_SAMPLES}]'
config['num_train_epochs'] = EPOCHS
config['per_device_train_batch_size'] = BATCH_SIZE
config['per_device_eval_batch_size'] = BATCH_SIZE
config['output_dir'] = str(OUTPUT_DIR)
config['model_name_or_path'] = str(BASE_TRAIN_DIR)
runtime_config = Path('/kaggle/working/finetune_mms_fa_runtime.json')
runtime_config.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
print(runtime_config.read_text(encoding='utf-8'))

## Prepare the Persian training checkpoint

The public inference checkpoint contains the generator. Fine-tuning also needs the original discriminator, so the upstream conversion utility downloads and combines both once.

In [ ]:
if not (BASE_TRAIN_DIR / 'config.json').exists():
    subprocess.run(
        [
            sys.executable,
            'convert_original_discriminator_checkpoint.py',
            '--language_code', 'fas',
            '--pytorch_dump_folder_path', str(BASE_TRAIN_DIR),
        ],
        cwd=REPO_DIR,
        check=True,
    )
print('Training checkpoint:', BASE_TRAIN_DIR)

## Fine-tune

Training checkpoints and TensorBoard logs are written under `/kaggle/working/persian-mms-tts`. Rerunning this cell resumes from the latest checkpoint when the output directory is preserved.

In [ ]:
env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'
subprocess.run(
    [
        'accelerate', 'launch',
        '--num_processes', '1',
        '--mixed_precision', 'fp16',
        'run_vits_finetuning.py', str(runtime_config),
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)

## Test the fine-tuned voice

In [ ]:
import soundfile as sf
from IPython.display import Audio, display
from transformers import AutoTokenizer, VitsModel

text = 'سلام، این یک نمونه از صدای فارسی آموزش‌دیده روی کگل است.'
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
model = VitsModel.from_pretrained(OUTPUT_DIR).to('cuda').eval()
inputs = tokenizer(text, return_tensors='pt').to('cuda')
with torch.no_grad():
    waveform = model(**inputs).waveform[0].cpu().numpy()

wav_path = Path('/kaggle/working/persian-test.wav')
sf.write(wav_path, waveform, model.config.sampling_rate)
display(Audio(filename=str(wav_path)))
print('Saved:', wav_path)

In [ ]:
import shutil

archive = shutil.make_archive('/kaggle/working/persian-mms-tts', 'zip', OUTPUT_DIR)
print('Download this file from the Kaggle Output panel:', archive)